# Tutorial 04 | SCOPAtlas — operator-regime atlas

Builds a stable-operator atlas on top of a fitted scQDiff drift model.
Start from any `adata` where `sqd.tl.fit_drift` has already been run,
or load a pre-trained model checkpoint.

**Typical workflow:**
```
sqd.pp.prepare_trajectory → sqd.tl.fit_drift → SCOPAtlas
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scanpy as sc
import scqdiff as sqd

from scqdiff.atlas import (
    StableOperatorAtlas,
    compute_operator_embedding,
    OperatorClustering,
)
from scqdiff.models.drift import DriftField, DriftConfig

OUTDIR = 'results/04_atlas/'
os.makedirs(OUTDIR, exist_ok=True)
print(f'scqdiff v{sqd.__version__}')

## Option A — Use a freshly fitted model (recommended)

Run the full pipeline and pass the returned model directly to the atlas.

In [ ]:
# ── Uncomment to run from scratch ────────────────────────────────────────
# adata = sc.datasets.paul15()
# sqd.pp.prepare_trajectory(adata, groupby='paul15_clusters', root='7MEP', n_hvg=2000)
# model = sqd.tl.fit_drift(adata, time_key='pseudotime', n_archetypes=5, n_epochs=5000)
# adata.write_h5ad(OUTDIR + 'adata_with_drift.h5ad')
# torch.save({'state_dict': model.state_dict(),
#             'cfg': model.cfg}, OUTDIR + 'drift_model.pt')
print("Uncomment the block above to run from scratch.")

## Option B — Load pre-trained model from checkpoint

In [ ]:
# ── Load adata and model ──────────────────────────────────────────────────
ADATA_PATH = OUTDIR + 'adata_with_drift.h5ad'   # produced by fit_drift above
MODEL_PATH  = OUTDIR + 'drift_model.pt'

if os.path.exists(ADATA_PATH) and os.path.exists(MODEL_PATH):
    adata      = ad.read_h5ad(ADATA_PATH)
    checkpoint = torch.load(MODEL_PATH, map_location='cpu')
    cfg        = checkpoint['cfg']
    X_pca      = torch.tensor(adata.obsm['X_pca'].astype('float32'))

    # Rebuild velocity prior tensors if needed
    from scqdiff.tl._drift import _pseudotime_velocity
    V_np = _pseudotime_velocity(adata.obsm['X_pca'],
                                adata.obs['pseudotime'].values)
    V_t  = torch.tensor(V_np)

    model = DriftField(cfg, X_ref=X_pca, V_ref=V_t)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    print("Loaded from checkpoint:", ADATA_PATH)
else:
    print("No checkpoint found. Run Option A first.")
    raise SystemExit

## 1. Build stable-operator atlas

In [ ]:
atlas = StableOperatorAtlas(
    adata      = adata,
    drift_model= model,
    use_rep    = 'X_pca',
    time_key   = 'pseudotime',
)
atlas.build(epsilon=0.1, threshold_unstable=0.1)

print("Operator regime counts:")
print(adata.obs['operator_regime'].value_counts())

## 2. Visualize operator metrics on UMAP

In [ ]:
metrics = ['lambda_max_plus', 'operator_regime']
sc.pl.umap(adata, color=metrics, ncols=2, save=False)

## 3. Operator-based clustering

In [ ]:
clusterer = OperatorClustering(adata)
clusterer.prepare_operator_features()
clusterer.cluster_operator_space(resolution=1.0)
clusterer.cluster_joint_space(alpha=0.5, resolution=1.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color='operator_clusters', ax=axes[0], show=False)
sc.pl.umap(adata, color='joint_clusters',    ax=axes[1], show=False)
plt.tight_layout(); plt.savefig(OUTDIR + 'clustering.pdf', dpi=150); plt.show()

## 4. Validate non-redundancy with cell types

In [ ]:
if 'paul15_clusters' in adata.obs.columns:
    validation = atlas.validate_nonredundancy(
        celltype_key='paul15_clusters',
    )
    print(validation)

    crosstab = pd.crosstab(adata.obs['paul15_clusters'],
                            adata.obs['operator_regime'],
                            normalize='index')
    plt.figure(figsize=(10, 5))
    sns.heatmap(crosstab, annot=True, fmt='.2f', cmap='Blues')
    plt.title('Cluster × regime overlap'); plt.tight_layout()
    plt.savefig(OUTDIR + 'celltype_regime_overlap.pdf', dpi=150); plt.show()

## 5. Save atlas

In [ ]:
atlas.save(OUTDIR + 'scopatlas.h5ad')
print(f"Atlas saved to {OUTDIR}scopatlas.h5ad")

## Instability Gene Analysis

Once `sqd.tl.fit_drift` has been run (Option A or B above), the instability scores are already stored in `adata.uns['scqdiff']`. Run the same analysis to identify which genes drive instability in your dataset.

In [ ]:
table = sqd.pl.instability_genes(
    adata,
    n_genes               = 15,
    sensitivity_threshold = 0.05,
    per_archetype         = True,
    save = OUTDIR + 'instability_genes.pdf',
)

In [ ]:
table.to_csv(OUTDIR + 'instability_genes_table.csv', index=False)
print(table.to_string(index=False))